In [0]:
# Imports
import logging
import requests
import time
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Catalog / Schema / Volume
catalog = "workspace"
schema = "ai_project"
volume = "raw_data"

# Base Volume Path
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"

# Docs Paths
docs_landing_path = vol_path + "docs/raw"

# Doc Sources Table
doc_sources_table = f"{catalog}.{schema}.doc_sources"

# %pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt
# dbutils.library.restartPython()

In [0]:
%run ./utils/logging_utils

In [0]:
# Fetch & clean docs from doc_sources
def clean_html(html_content):
    """
    Strips HTML tags and cleans raw page content into plain text.

    Args:
        html_content (str): Raw HTML string from requests
    
    Returns:
        str: Cleaned plain text
    """
    soup = BeautifulSoup(html_content, "html.parser")

    # Remove navigation, headers, footers, scripts, styles
    for element in soup(["nav", "header", "footer", "script", "style", "aside"]):
        element.decompose()

    # Extract plain text
    text = soup.get_text(separator="\n")

    # Clean up excessive whitespace and blank lines
    lines = [line.strip() for line in text.splitlines()]
    cleaned = "\n".join(line for line in lines if line)

    return cleaned


def fetch_docs():
    """
    Queries doc_sources for active URLs, fetches each page,
    cleans the HTML, and saves as .txt files to docs/raw Volume.
    """
    # Query active sources
    sources_df = spark.sql(f"""
        SELECT doc_id, url, title, topic
        FROM {doc_sources_table}
        WHERE active = true
    """)

    sources = sources_df.collect()

    if not sources:
        logger.warning("⚠️ No active sources found in doc_sources.")
        return

    logger.info(f"✅ Found {len(sources)} active sources to fetch.")

    for row in sources:
        doc_id = row["doc_id"]
        url = row["url"]
        title = row["title"]
        topic = row["topic"]

        # Generate a clean file name from topic and title
        file_name = f"{topic}_{title}.txt".replace(" ", "_").replace("/", "-")
        file_path = f"{docs_landing_path}/{file_name}"

        try:
            logger.info(f"🌐 Fetching: {url}")

            response = requests.get(url, timeout=15)
            response.raise_for_status()
            response.encoding = "utf-8"

            # Clean HTML content
            cleaned_text = clean_html(response.text)

            if not cleaned_text.strip():
                raise ValueError("Cleaned content is empty — page may be JS-rendered.")

            # Calculate file size
            file_size_kb = len(cleaned_text.encode("utf-8")) / 1024

            # Write to Volume
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(cleaned_text)

            logger.info(f"✅ Saved: {file_name} ({file_size_kb:.2f} KB)")

            # Log success
            write_ingestion_log(
                doc_id=doc_id,
                url=url,
                status="SUCCESS",
                file_name=file_name,
                file_size_kb=round(file_size_kb, 2)
            )

        except Exception as e:
            logger.error(f"❌ Failed to fetch {url}: {e}")

            # Log failure
            write_ingestion_log(
                doc_id=doc_id,
                url=url,
                status="FAILED",
                error_message=str(e)
            )

    logger.info("🏁 Doc fetching complete.")

In [0]:
# Run
fetch_docs()

In [0]:
dbutils.fs.ls("/Volumes/workspace/ai_project/raw_data/docs/raw/")

In [0]:
with open("/Volumes/workspace/ai_project/raw_data/docs/raw/Delta_Lake_Delta_Lake_Tables_Overview.txt", "r", encoding="utf-8") as f:
    print(f.read())